# Week 2 Day 4 Lab - Scoring the run and carrying on after a crash

*Arthur's version of the assignment: a `score_run` node that brings the swarm back together, and a crash in the middle of a run, to see which claims LangGraph works again.*

This builds on [the lab](../../labs/week2day4.ipynb), so run that first. The dispatcher is in [swarm_graph.py](swarm_graph.py), so that this notebook and a separate process can build the same graph. The checkpoints go in a schema of their own in Supabase, `lab_swarm`.

## 3 parts:

PART 1: score_run, tried on stand-ins

PART 2: The swarm, scored by the graph

PART 3: A crash in the middle of a run

In [ ]:
import inspect
import re
import subprocess
import sys
import threading
import time
from datetime import datetime

import pandas as pd
import plotly.express as px
from IPython.display import Image, display
from langfuse import get_client
from langfuse.langchain import CallbackHandler

import swarm_graph
from swarm_graph import build_dispatcher, claims, connect, labels, postgres_checkpointer, recorded, run_config, run_scores, score_runs, score_table, work_claim

## Part 1: score_run, tried on stand-ins

The state gets a `scores` field, and `score_run` fills it from every claim's result, with the lab's `score_table` and `run_scores`. The edge from `work_claim` goes to `score_run` in place of `END`.

In [ ]:
print(inspect.getsource(swarm_graph.score_run))
print(inspect.getsource(swarm_graph.build_dispatcher))

Every `Send` puts its `work_claim` in the same step. LangGraph waits for all of them, adds their results together with the `operator.add` reducer, and then runs `score_run` once in the next step, even though 25 tasks lead to it. `score_runs` counts how many times it ran, and on how many results.

In [ ]:
display(Image(build_dispatcher().get_graph().draw_mermaid_png()))

### Stand-ins with known answers

The lab's stand-in returns no decisions, so there's nothing to score. These two return answers we know the scores for, which tests the scorer before we spend anything on a model.

- **The perfect stand-in** gives the handler's decision and payout on every claim. Its accuracy must be 100% and its cost of mistakes $0.
- **The cautious stand-in** refers every claim. Its straight-through rate must be 0%, and each claim the handlers decided costs a needless referral of $50.

In [ ]:
def perfect_stand_in(claim_id):
    """Give the handler's answer after a moment."""
    started = datetime.now()
    time.sleep(0.5)
    label = labels[claim_id]
    return {"results": [{"claim_id": claim_id, "decision": label["decision"], "payout": label["payout"], "rationale": "The handler's answer", "cost": 0.0, "started": started, "finished": datetime.now()}]}


def cautious_stand_in(claim_id):
    """Refer every claim after a moment."""
    started = datetime.now()
    time.sleep(0.5)
    return {"results": [{"claim_id": claim_id, "decision": "refer", "payout": 0.0, "rationale": "Referred", "cost": 0.0, "started": started, "finished": datetime.now()}]}


decided_by_handlers = sum(label["decision"] != "refer" for label in labels.values())
expected = {
    "Perfect": {"accuracy": 1.0, "cost_of_mistakes": 0.0, "straight_through": decided_by_handlers / len(labels)},
    "Cautious": {"accuracy": (len(labels) - decided_by_handlers) / len(labels), "cost_of_mistakes": 50.0 * decided_by_handlers, "straight_through": 0.0},
}

rows = []
for name, stand_in in [("Perfect", perfect_stand_in), ("Cautious", cautious_stand_in)]:
    score_runs.clear()
    scores = build_dispatcher(work=stand_in).invoke({}, {"max_concurrency": 25})["scores"]
    for score, value in expected[name].items():
        assert abs(scores[score] - value) < 1e-9, f"{name}: {score} is {scores[score]}, expected {value}"
    rows.append({"stand_in": name, "score_run_ran": len(score_runs), "on_results": score_runs[0], **{score: scores[score] for score in ["claims", "accuracy", "cost_of_mistakes", "straight_through", "seconds"]}})

pd.DataFrame(rows)

Both stand-ins score exactly as expected, and `score_run` ran once per run, on all 25 results.

## Part 2: The swarm, scored by the graph

The real claim agents, 5 at once, as in the lab. The scores come back in the state, and we check them against the lab's own way of scoring, outside the graph, then send them to the run's trace in Langfuse.

In [ ]:
langfuse = get_client()
langfuse_handler = CallbackHandler()

recorded.clear()
score_runs.clear()
result = build_dispatcher().invoke({}, {"max_concurrency": 5, "callbacks": [langfuse_handler], "run_name": "Swarm of 5, scored"})

outside = run_scores(score_table(result["results"]))
assert all(abs(result["scores"][name] - float(value)) < 1e-9 for name, value in outside.items())
print(f"score_run ran {len(score_runs)} time, on {score_runs[0]} results, and agrees with scoring outside the graph")

for name, value in result["scores"].items():
    langfuse.create_score(trace_id=langfuse_handler.last_trace_id, name=name, value=value)

pd.Series(result["scores"]).round(3)

## Part 3: A crash in the middle of a run

We compile the dispatcher with Day 3's `PostgresSaver` and run it on a thread of its own. The assignment restarts the kernel while the claim agents are working. To make the crash the same every time, we run the dispatcher in a separate process and kill it part way through. That's what a kernel restart does: the process is gone, and only what's in Supabase is left.

**To do it by hand instead:** run `uv run community_contributions/arthur_jzl/swarm_graph.py swarm-by-hand` in a terminal, press Ctrl+C after a few claims have finished, and call `resume("swarm-by-hand", max_concurrency=5)` below.

`crash_run` starts the process and kills it 15 seconds after its first claim starts. `resume` reads the thread back from Supabase, lists the results saved before the crash, and carries the run on in this notebook with `invoke(None, config)`, noting every claim it works again.

In [ ]:
connection = connect()
checkpointer = postgres_checkpointer(connection)
CRASH_AFTER = 15


def crash_run(thread_id, *options):
    """Run the swarm in another process on a fresh thread, and kill it part way through. Returns what it printed."""
    checkpointer.delete_thread(thread_id)
    process = subprocess.Popen(
        [sys.executable, "swarm_graph.py", thread_id, *options],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", bufsize=1,
    )
    lines, first_claim = [], threading.Event()

    def read():
        for line in process.stdout:
            lines.append(line.rstrip())
            if line.startswith("Working"):
                first_claim.set()

    reader = threading.Thread(target=read, daemon=True)
    reader.start()
    first_claim.wait(timeout=120)
    time.sleep(CRASH_AFTER)
    process.kill()
    process.wait()
    reader.join(timeout=5)
    return lines


def before_crash(lines):
    """The claims the killed process started, and the ones it finished with their cost."""
    started = [line.split()[1] for line in lines if line.startswith("Working")]
    finished = {match[1]: float(match[2]) for line in lines if (match := re.match(r"Finished (\S+): .*cost \$([\d.]+)", line))}
    return started, finished


def resume(thread_id, max_concurrency=None, at_once=None):
    """Carry the thread on in this process. Returns the claims saved before the crash, the ones worked again, and the result."""
    worked_again = []

    def tracked(claim_id):
        worked_again.append(claim_id)
        return work_claim(claim_id)

    recorded.clear()
    score_runs.clear()
    dispatcher = build_dispatcher(work=tracked, checkpointer=checkpointer, at_once=at_once)
    config = run_config(thread_id, max_concurrency, at_once)
    state = dispatcher.get_state(config)
    tasks = [task for task in state.tasks if task.name == "work_claim"]
    saved = [result["claim_id"] for task in tasks if task.result for result in task.result["results"]]
    print(f"Saved in Supabase: {len(saved)} results, {len(tasks) - len(saved)} claims still to work")
    if not state.next:
        print("Nothing saved to carry on from: the run would have to start again from the beginning")
        return saved, worked_again, None
    return saved, worked_again, dispatcher.invoke(None, config, durability="sync")


crashes = {}


def crash_and_resume(name, thread_id, durability, max_concurrency=None, at_once=None):
    """Crash a run, carry it on, and say which claims were worked again."""
    options = ["--durability", durability] + (["--at-once", str(at_once)] if at_once else ["--max-concurrency", str(max_concurrency)])
    started, finished = before_crash(crash_run(thread_id, *options))
    print(f"Before the crash: {len(started)} claims started, {len(finished)} finished")
    saved, worked_again, result = resume(thread_id, max_concurrency, at_once)
    in_flight = [claim_id for claim_id in started if claim_id not in finished]
    if result:
        print(f"Worked again: {len(worked_again)} claims")
        print(f"  of the {len(finished)} finished before the crash: {len(set(worked_again) & set(finished))}")
        print(f"  of the {len(in_flight)} in flight at the crash: {len(set(worked_again) & set(in_flight))}")
        print(f"  of the {len(claims) - len(started)} never started: {len(set(worked_again) - set(started))}")
        print(f"score_run ran {len(score_runs)} time, on {score_runs[0]} results")
    else:
        print(f"Starting again would work all {len(claims)} claims, including the {len(finished)} finished before the crash")
    crashes[name] = {"started": started, "finished": finished, "saved": saved, "worked_again": worked_again, "result": result}

### Crash 1: 5 at once, the lab's way

`max_concurrency=5` and `durability="sync"`, as in the lab and on Day 3.

In [ ]:
crash_and_resume("max_concurrency=5, sync", "swarm-lab", "sync", max_concurrency=5)

### Why the finished claims weren't saved

LangGraph runs the nodes on a pool of threads, and saves each task's result on the same pool. `max_concurrency` sets the size of that pool. With 5 threads and 25 claims, 5 claims run and 20 wait in the pool's queue. When a claim finishes, the job of saving its result joins the back of the queue, behind the claims still waiting. So nothing is saved until the last claims have started, and a crash before then loses every result.

I measured it with a stand-in at 1 second a claim while building this. With `max_concurrency=5`, no results were saved in the first 4 seconds, though 15 claims had finished. With `max_concurrency=25`, 11 were saved after 2 seconds.

**The fix:** cap the claims with a gate inside the node, a semaphore that lets 5 claims through at a time, and give the pool a thread for every claim and a few to spare, which `run_config` does. The claims waiting at the gate hold their threads, and the spare threads save each result as soon as its claim finishes.

In [ ]:
print(inspect.getsource(swarm_graph.gated))
print(inspect.getsource(swarm_graph.run_config))

### Crash 2: 5 at once, with the gate

In [ ]:
crash_and_resume("gate of 5, sync", "swarm-gate-sync", "sync", at_once=5)

In [ ]:
def show_timeline(results, saved, title):
    """Each claim as a bar, coloured by whether it ran before the crash or after."""
    table = pd.DataFrame(results).assign(run=lambda t: t["claim_id"].map(lambda claim_id: "before the crash" if claim_id in saved else "after the crash"))
    fig = px.timeline(table, x_start="started", x_end="finished", y="claim_id", color="run", title=title)
    fig.update_yaxes(autorange="reversed")
    fig.show()


gate_crash = crashes["gate of 5, sync"]
show_timeline(gate_crash["result"]["results"], gate_crash["saved"], "Gate of 5, sync: saved before the crash, worked after it")
pd.Series(gate_crash["result"]["scores"]).round(3)

### Extra: the three durability settings

`durability` says when LangGraph saves what a step has done:

- **`sync`** saves each task's result before the run goes on.
- **`async`**, the default, saves in the background while the run goes on.
- **`exit`** saves only when the run ends, whether it finishes or fails. A killed process never gets there.

We crash the run with the gate in the same way with `async` and with `exit`.

In [ ]:
for durability in ["async", "exit"]:
    print(f"--- gate of 5, {durability}")
    crash_and_resume(f"gate of 5, {durability}", f"swarm-gate-{durability}", durability, at_once=5)

### Extra: what the crash cost

The spend on a claim that finished before the crash is kept if its result was saved, and wasted if it wasn't. The spend on a claim in flight at the crash is always wasted. We can't see how far those claim agents got, so `in_flight_up_to` counts a whole claim's average cost for each, which makes it an upper bound.

When nothing was saved, carrying on means working all 25 claims again. That run isn't made here, and `spend_to_finish` estimates it at the average cost per claim.

In [ ]:
runs = [crash["result"] for crash in crashes.values() if crash["result"]]
average_cost = sum(run["scores"]["llm_cost"] for run in runs) / sum(run["scores"]["claims"] for run in runs)

rows = []
for name, crash in crashes.items():
    in_flight = [claim_id for claim_id in crash["started"] if claim_id not in crash["finished"]]
    result = crash["result"]
    rows.append({
        "run": name,
        "finished_before_crash": len(crash["finished"]),
        "saved": len(crash["saved"]),
        "worked_again": len(crash["worked_again"]) if result else len(claims),
        "spend_kept": sum(cost for claim_id, cost in crash["finished"].items() if claim_id in crash["saved"]),
        "finished_but_lost": sum(cost for claim_id, cost in crash["finished"].items() if claim_id not in crash["saved"]),
        "in_flight_up_to": len(in_flight) * average_cost,
        "spend_to_finish": sum(item["cost"] for item in result["results"] if item["claim_id"] in crash["worked_again"]) if result else len(claims) * average_cost,
    })

pd.DataFrame(rows).set_index("run").round(4)

### The claim agents save their steps too

`work_claim` calls the claim agent, and the claim agent is a LangGraph graph too. When the dispatcher has a checkpointer, LangGraph hands it on, so each claim agent saves its own steps on the dispatcher's thread, under a namespace of its own. While building this I checked the claim agents that were worked again after a crash: each one ended with a single conversation of 6 messages, with the claim's prompt once, so working a claim again doesn't leave a doubled conversation behind.

### Decisions that live in memory

`recorded` lives in the process's memory, so the crash wiped it. A claim agent that recorded its decision and was killed before its node returned is worked again, and "decisions are final" doesn't stop a second decision, because the first is gone. In the product, `record_decision` writes to the backend's database, which refuses a second decision on the same claim, so a claim worked again after a crash can't be decided twice. Durable state in the graph isn't enough on its own: the tools' side effects need to be durable too.

### My findings

From the run above, with GPT-6 Luna at `low` through OpenRouter, on October 28, 2026. The whole notebook costs about 5 cents.

**`score_run` runs once, after every claim agent.** With both stand-ins it ran once, on all 25 results, and the scores came out exactly as expected. The perfect stand-in scored 100% accuracy and $0 of mistakes. The cautious one scored 0% straight-through and $950 of mistakes: 19 needless referrals at $50. On the real swarm it ran once again, and its scores matched the lab's scoring outside the graph to the last digit.

**Which claims does LangGraph work again? With the lab's settings, all of them.** With `max_concurrency=5`, the crash came after 5 claims had finished, but none of their results had been saved, so the resumed run worked all 25 claims again. LangGraph saves a task's result on the same pool of threads that runs the nodes, and `max_concurrency` sets the pool's size. Each result waits to be saved behind the claims still queued to start, so nothing is saved until the last claims have started.

**With a gate in the node, only the unfinished claims.** Capping the claims with a semaphore in the node, and giving the pool a thread for every claim, saved each result as its claim finished. After the crash, the 5 finished claims came back from Supabase, and LangGraph worked only the other 20: the 5 in flight at the crash and the 15 that had never started. `score_run` then ran once, on all 25. Still no more than 5 claims ran at once.

**Durability:** `async` behaved like `sync` here: 7 results saved, 18 claims worked again. With `async` the saves happen in the background, so a result being saved at the very moment of the crash could be lost, but there's plenty of time for the saves in a claim agent's 7 seconds. With `exit`, nothing was saved, so there was nothing to carry on from: the 9 claims finished before the crash were lost, and the run would have to start again.

**What the crash cost:** with the gate and `sync`, finishing the run cost $0.0062, against $0.0085 for working all 25 again. These amounts are small because each claim costs about a third of a cent, but the share grows with the run. The bigger cost is time: with the lab's settings, a crash near the end of a long run means working the whole run again.

**Two more things a crash shows:**
- **The claim agents save their steps too.** LangGraph hands the dispatcher's checkpointer to the claim agent inside `work_claim`, so each agent saves its own steps on the dispatcher's thread, under a namespace of its own. A claim agent worked again after a crash ended with a single, normal conversation.
- **`recorded` is in memory, so the crash wipes it.** A claim that recorded its decision just before the crash is worked again with nothing to stop a second decision. In the product, the backend's database has to refuse a second decision on the same claim, because durable state in the graph doesn't cover the tools' side effects.